In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
import scipy.special as sp
from SamplableSet import SamplableSet
import ast
import json
import os
from math import comb, log

colors_7 = ['#5c3e81', '#944087', '#c6447f', '#ed536c', '#ff7151', '#ff9931', '#ffc400']

## Generating the structural distributions

In [2]:
def P_n_act(n_act, n, hom, A):
    """
    Group composition distribution, describing the probability of having n_act active nodes in a
    group of size n, given homophily hom and proportion of active nodes in the population, D. 
    Consists of a mixture of two binomial distributions.
    
    Inputs
    n_act: number of active nodes. Has to be between 0 and n
    n: size of the group
    hom: homophily (0.0 is no homophily, 1.0 is total homophily)
    A: fraction of active nodes in the population
    
    Output
    The composition probability
    """
    binom = sp.binom(n, n_act)
    left_term = binom*(hom+A*(1-hom))**n_act*((1-hom)*(1-A))**(n-n_act)
    right_term = binom*((1-hom)*A)**n_act*(1-A + hom*A)**(n-n_act)
    return A*left_term + (1-A)*right_term

def P_n_n_act(n_act, n, s, tau): 
    """
    Response distribution: outputs the probability that a group of size n is active if it contains n_act active nodes.
    
    s: controls the gradient of the function around its threshold. For s -> infty, this function
    becomes a Heaviside step function.
    
    tau: controls the position of the point where the probability is 0.5.
    """
    if tau==1:
        tau = 1-1e-8
    elif tau==0:
        tau = 1e-8
    r = - np.log(2)/np.log(tau)
    if n_act == 0:
        return 0
    elif n_act == n:
        return 1
    else:
        n_scaled = n_act/n
        return (1-n_scaled**r)**(-s)/((1-n_scaled**r)**(-s) + n_scaled**(-s*r))

## Generating the network structure

### Generating the node list

In the "node_stubs" list, each node appears as many times as its membership. Each instance of the node represents a stub.

In [3]:
def generate_node_stubs(N, A, m_min=2, m_max=80, g_m = None, gamma_m=3.2):
    """
    This function returns two ordered list of integers, where each integer represents a node, 
    and its multiplicity corresponds to the membership of that node. Each instance of a node represents a stub.
    One list contains active nodes, the other, passive nodes.
    
    Inputs
    gamma_m: abs val of the exponent of the (power-law) membership distribution
    m_min: minimum membership
    m_max: maximum membership
    N: number of nodes
    A: expected fraction of active nodes in the population
    
    Outputs
    active_node_stubs: the above-mentioned list of active node stubs
    passive_node_stubs: self-explanatory
    """
    active_node_stubs = []
    passive_node_stubs = []
    
    if g_m is None:
        membs = np.arange(m_min, m_max + 1)
        raw = membs.astype(float) ** (-gamma_m)
        raw /= raw.sum()
        g_m = raw
    m_list = np.random.choice(range(m_min, m_max+1), size=N, p=g_m) #Draw the memberships
    activities = np.array([0] * int(N * A) + [1] * (N - int(N * A)))
    np.random.shuffle(activities) #Draw the activites
    for i in range(N):
        if activities[i] == 0:
            active_node_stubs.extend([i for j in range(m_list[i])])
        else:
            passive_node_stubs.extend([i for j in range(m_list[i])])
            
    active_node_stubs = np.random.permutation(active_node_stubs).tolist()
    passive_node_stubs = np.random.permutation(passive_node_stubs).tolist()

    return active_node_stubs, passive_node_stubs

### Generating group list

The first cell generates the group size distribution. The second cell generates the group list. The group list consists of $n_i$ instances of the index of each group, where $n_i$ is the size of group $i$. The if clauses in the loop make sure that the group list and node list are of the same length for the stub matching process. The stub matching is made through shuffling the group list.

In [4]:
def generate_group_stubs(
        active_nodes, passive_nodes,
        h, A,
        n_min=2, n_max=80,
        gamma_n=2.6, p_n=None,
        output_group_size=True,
        output_n_act=True):
    """
    Generate group stubs with homophily, while strictly respecting:
      - total active stubs  = len(active_nodes)
      - total passive stubs = len(passive_nodes)
      - group sizes n in [n_min, n_max]
      - compositions drawn from P_n_act, truncated to feasible (n_act, n-n_act).

    h: 0 -> random mixing, 1 -> full homophily.
    A: fraction of active nodes in the population.
    """

    active_remaining = len(active_nodes)
    passive_remaining = len(passive_nodes)

    if p_n is None:
        sizes = np.arange(n_min, n_max + 1)
        raw = sizes.astype(float) ** (-gamma_n)
        raw /= raw.sum()
        p_n = raw
    else:
        sizes = np.arange(n_min, n_min + len(p_n))

    group_size = []
    n_act_list = []

    while True:
        total_remaining = active_remaining + passive_remaining

        if total_remaining == 0:
            break

        # If we have fewer stubs than n_min, fold them into the last group
        if total_remaining < n_min:
            extra = total_remaining
            if extra > 0:
                g_id = len(group_size) - 1
                n_old = group_size[g_id]
                a_old = n_act_list[g_id]

                # Feasible extra active stubs:
                n_act_min_extra = max(0, extra - passive_remaining)
                n_act_max_extra = min(extra, active_remaining)
                n_acts_extra = np.arange(n_act_min_extra, n_act_max_extra + 1)

                # Truncated distribution induced by P_n_act for the updated group
                weights = np.array(
                    [P_n_act(a_old + a_e, n_old + extra, h, A) for a_e in n_acts_extra],
                    dtype=float
                )
                wsum = weights.sum()
                if wsum <= 0:
                    weights[:] = 1.0 / len(weights)
                else:
                    weights /= wsum

                a_extra = int(np.random.choice(n_acts_extra, p=weights))

                # Update last group
                group_size[g_id] = n_old + extra
                n_act_list[g_id] = a_old + a_extra

                # Consume remaining stubs
                active_remaining  -= a_extra
                passive_remaining -= (extra - a_extra)

            break

        # Otherwise, we can create a new group with size n in [n_min, min(n_max, total_remaining)]
        max_n = min(n_max, total_remaining)
        possible_ns = np.arange(n_min, max_n + 1)
        probs = np.array([p_n[n - n_min] for n in possible_ns], dtype=float)
        probs /= probs.sum()
        n = int(np.random.choice(possible_ns, p=probs))

        # Feasible n_act for this group:
        #   0 <= n_act <= active_remaining
        #   0 <= n - n_act <= passive_remaining
        n_act_min = max(0, n - passive_remaining)
        n_act_max = min(n, active_remaining)

        # If infeasible, force this group to take all stubs
        if n_act_min > n_act_max:
            n = total_remaining
            n_act_min = max(0, n - passive_remaining)
            n_act_max = min(n, active_remaining)

        n_acts = np.arange(n_act_min, n_act_max + 1)

        # Truncated P_n_act on feasible n_acts
        weights = np.array([P_n_act(n_act, n, h, A) for n_act in n_acts], dtype=float)
        wsum = weights.sum()
        if wsum <= 0:
            weights[:] = 1.0 / len(weights)
        else:
            weights /= wsum

        n_act = int(np.random.choice(n_acts, p=weights))

        group_size.append(n)
        n_act_list.append(n_act)

        active_remaining  -= n_act
        passive_remaining -= (n - n_act)

    # Build stub lists
    active_group_stubs = []
    passive_group_stubs = []
    for g_id, (n, n_act) in enumerate(zip(group_size, n_act_list)):
        if n_act > 0:
            active_group_stubs.extend([g_id] * n_act)
        if n_act < n:
            passive_group_stubs.extend([g_id] * (n - n_act))

    active_group_stubs.sort()
    passive_group_stubs.sort()

    if output_group_size and output_n_act:
        return active_group_stubs, passive_group_stubs, group_size, n_act_list
    elif output_group_size:
        return active_group_stubs, passive_group_stubs, group_size
    elif output_n_act:
        return active_group_stubs, passive_group_stubs, n_act_list
    else:
        return active_group_stubs, passive_group_stubs


In [7]:
def get_group_indices(active_group_stubs, passive_group_stubs):
    active_group_indices = {} #Using a dictionary for the indices of the nodes in the lists.
    passive_group_indices = {}

    for i, group in enumerate(active_group_stubs):
        if group not in active_group_indices:
            active_group_indices[group] = [i, i]  # [first_index, last_index]
        else:
            active_group_indices[group][1] = i  # update last_index

    for i, group in enumerate(passive_group_stubs):
        if group not in passive_group_indices:
            passive_group_indices[group] = [i, i]  # [first_index, last_index]
        else:
            passive_group_indices[group][1] = i  # update last_index
    
    return active_group_indices, passive_group_indices

In [8]:
def active_group_nodes(group, active_node_stubs, active_group_indices):
    """
    Outputs which active nodes belong to the given group
    """
    if group in active_group_indices:
        node_list = active_node_stubs[active_group_indices[group][0]:active_group_indices[group][1]+1]
    else:
        node_list = []
        
    return node_list

def passive_group_nodes(group, passive_node_stubs, passive_group_indices):
    """
    Outputs which passive nodes belong to the given group
    """
    if group in passive_group_indices:
        node_list = passive_node_stubs[passive_group_indices[group][0]:passive_group_indices[group][1]+1]
    else:
        node_list = []
    
    return node_list

def group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices):
    return active_group_nodes(group, active_node_stubs, active_group_indices) + passive_group_nodes(group, passive_node_stubs, passive_group_indices)

In [9]:
def get_group_activity(group_size, n_act_list, s, tau):
    # Vectorize over group_size and calculate P_n_n_act for all nodes
    number_of_groups = len(group_size)

    # Precompute the probabilities for all groups at once
    P_values = [P_n_n_act(n_act_list[i], group_size[i], s, tau) for i in range(number_of_groups)]

    # Generate random values at once for all nodes
    
    # Use a vectorized condition to classify groups as active or passive
    active_groups = []
    passive_groups = []
    for i in range(number_of_groups):
        activity = random.random()
        if activity <= P_values[i]:
            active_groups.append(i)
        else:
            passive_groups.append(i)
    
    return active_groups, passive_groups

In [12]:
def get_neighbors_sets(active_groups, passive_groups, active_node_stubs, active_group_indices, passive_node_stubs, 
                       passive_group_indices, N):
    active_neighbors_set = [[] for i in range(N)]
    passive_neighbors_set = [[] for i in range(N)]

    for group in active_groups:
        nodes = group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices)
        for node in nodes:
            active_neighbors_set[node].extend([neighbor for neighbor in nodes if neighbor != node])

    for group in passive_groups:
        nodes = group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices)
        for node in nodes:
            passive_neighbors_set[node].extend([neighbor for neighbor in nodes if neighbor != node])

    return active_neighbors_set, passive_neighbors_set

### Running the simulation

In [13]:
def MC_simulation(beta, eff, active_degrees, passive_degrees, active_neighbors_set, passive_neighbors_set,
                  method='HR', get_times = False, output_states = False, batch_size = int(2e6), sample_size=1000, 
                  burn_in_time_constant=2e6, init_prev = 0.2, min_interval=5, savefile = None):
    
    passive_neighbors_set = [np.array(neigh, dtype=np.int32) for neigh in passive_neighbors_set]
    active_neighbors_set = [np.array(neigh, dtype=np.int32) for neigh in active_neighbors_set]
    
    assert len(passive_neighbors_set) == len(passive_degrees)
    assert len(active_neighbors_set) == len(active_degrees)
    
    if method != 'SQS' and method != 'HR':
        raise ValueError('Method must be SQS or HR')
    
    states = []
    eff_degrees = [passive_degrees[i] + eff*active_degrees[i] for i in range(N)]
    weights = [1+beta*eff_degrees[node] for node in range(N)]
    
    if method == 'HR':
        mx = max(eff_degrees)
        hubs = [i for i in range(N) if eff_degrees[i] == mx]

    #Setting the initial conditions
    node_states = [np.random.choice(2, p=[1-init_prev, init_prev]) for i in range(N)]
    infected_nodes = [i for i in range(N) if node_states[i] == 1]
    infected_weights = [weights[i] for i in infected_nodes]
    nodes_weights = list(zip(infected_nodes, infected_weights))
    s = SamplableSet(1, 1+beta*mx, nodes_weights)

    t = 0

    infected_count = sum(node_states)

    # Pre-generate random values to reduce redundant calls to np.random.rand()
    rand_vals = np.random.rand(batch_size).reshape((int(batch_size/2), 2))  # Generate in batches

    # Set up the index to track the random values
    rand_idx = 0
    step = 0

    total_prob = s.total_weight()
    burn_in_time = burn_in_time_constant/total_prob
    
    if method == 'SQS':
        history = []
        
    if get_times:
        times = []

    while len(states) < sample_size:

        # Use a batch of random values at once
        if rand_idx >= int(batch_size/2):
            rand_vals = np.random.rand(batch_size).reshape((int(batch_size/2), 2))
            rand_idx = 0
        r1, r2 = rand_vals[rand_idx]
        rand_idx += 1  # Move to the next batch

        tau = -np.log(r1) / total_prob
        t += tau

        chosen_node, weight = s.sample()

        if r2 < 1 / weight:
            if node_states[chosen_node] == 1:
                node_states[chosen_node] = 0
                infected_count -= 1  # Decrement infected count
                del s[chosen_node]
            
        elif r2 < (1 + beta * passive_degrees[chosen_node]) / weight:
            chosen_neighbor = np.random.choice(passive_neighbors_set[chosen_node])

            # Only update node state if it's not already infected
            if node_states[chosen_neighbor] == 0:
                node_states[chosen_neighbor] = 1
                infected_count += 1  # Increment infected count
                s.insert(chosen_neighbor, weights[chosen_neighbor])

        else:
            chosen_neighbor = np.random.choice(active_neighbors_set[chosen_node])

            # Only update node state if it's not already infected
            if node_states[chosen_neighbor] == 0:
                node_states[chosen_neighbor] = 1
                infected_count += 1  # Increment infected count
                s.insert(chosen_neighbor, weights[chosen_neighbor])
            
            
        if method == 'SQS':
            history.append(node_states.copy())
            if len(history) == 201:
                history = history[1:]

        if infected_count == 0:
            if method == 'HR':
                hub = np.random.choice(hubs)
                node_states[hub] = 1
                s.insert(hub, weights[hub])
                infected_count += 1
            elif method == 'SQS':
                node_states = np.random.choice(history)
                infected_nodes = [i for i in range(N) if node_states[i] == 1]
                infected_weights = [weights[i] for i in infected_nodes]
                nodes_weights = zip(infected_nodes, infected_weights)
                s = SamplableSet(1, 1+beta*mx, nodes_weights)
                step = np.random.randint(low=min_interval, high=10*min_interval+1)
        
        total_prob = s.total_weight()

        if t > burn_in_time and step <= 0:
            infected_nodes = [i for i in range(N) if node_states[i] == 1]
            states.append(np.array(infected_nodes, dtype=np.int32))
            if get_times:
                times.append(t)

        if step > 0:
            step -= 1
        elif step == 0:
            step = np.random.randint(low=10, high=51)
            
    if savefile is not None:
        np.savez_compressed(
                savefile,
                **{f"s{i}": states[i] for i in range(len(states))}
            )
    if output_states:
        return states

### Generating the group size and membership distributions

Here, the distributions follow power laws with the exponents given above. The cell would have to be changed for other distributions.

In [14]:
n_min = 2 # minimum group size
n_max = 70 # maximum group size
m_min = 2 # minimum membership
m_max = 70 # maximum membership

gamma_n = 2.6
gamma_m = 3.2

g_m = np.asarray([i**(-gamma_m) for i in range(m_min, m_max+1)])
g_m_norm = sum(g_m)
g_m /= g_m_norm # Normalizing to ensure the probabilities sum to 1

p_n = np.asarray([i**(-gamma_n) for i in range(n_min, n_max+1)])
p_n_norm = sum(p_n)
p_n /= p_n_norm # idem

# Generating networks

In [15]:
N=200000
A=0.5
tau=0.3
s=10

In [ ]:
for it in tqdm(range(550)):
    
    hom = (int(it/50))/10
    
    active_node_stubs, passive_node_stubs = generate_node_stubs(N, A, m_min=m_min, m_max=m_max, g_m = g_m)
    
    active_group_stubs, passive_group_stubs, group_size, n_act_list = generate_group_stubs(active_node_stubs, 
                                                                                           passive_node_stubs, hom, A, 
                                                                                           n_min=n_min, n_max=n_max, gamma_n=gamma_n, p_n=p_n)
    
    number_of_groups = len(group_size)
    
    active_groups, passive_groups = get_group_activity(group_size, n_act_list, s=s, tau=tau)
    
    active_group_indices, passive_group_indices =  get_group_indices(active_group_stubs, passive_group_stubs)
    
    active_neighbors_set, passive_neighbors_set = get_neighbors_sets(active_groups, passive_groups, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices, N)
    
    active_degrees = [len(elem) for elem in active_neighbors_set]
    passive_degrees = [len(elem) for elem in passive_neighbors_set]
    
    data = open("AME_MC_data/Structures/Homophily_{0}/structure_{1}A.txt".format(10*hom, it%50), "w")
    data.write("n_min = {0}, n_max = {1}, m_min = {2}, m_max = {3}, gamma_m = {4}, gamma_n = {5}\n".format(n_min, n_max, m_min, m_max, gamma_m, gamma_n))
    data.write("tau = {0}, A = {1}, sigma = {2}\n".format(tau, A, s))
    data.write("active_node_stubs = {}\n".format(active_node_stubs))
    data.write("passive_node_stubs = {}\n".format(passive_node_stubs))
    data.write("group_size = {}\n".format(group_size))
    data.write("n_act_list = {}\n".format(n_act_list))
    data.write("active_group_stubs = {}\n".format(active_group_stubs))
    data.write("passive_group_stubs = {}\n".format(passive_group_stubs))
    data.write("active_neighbors_set = {}\n".format(active_neighbors_set))
    data.write("passive_neighbors_set = {}\n".format(passive_neighbors_set))
    data.write("active_groups = {}\n".format(active_groups))
    data.write("passive_groups = {}".format(passive_groups))

    data.close()

  5%|▌         | 28/550 [08:39<2:38:00, 18.16s/it]

# Generating time series

In [ ]:
hom = 0.0
mult_list = [i/2 for i in range(1, 17)]
for mult in tqdm(mult_list):
    for it in range(50):
        with open("AME_MC_data/Structures/Homophily_{0}/structure_{1}.txt".format(10*hom, it), "r") as structure:
            i = 0
            for line in structure:
                if i==0 or i==1:
                    pass
                elif i==2:
                    pass
                elif i==3:
                    pass
                elif i==4:
                    pass
                elif i==5:
                    pass
                elif i==6:
                    pass
                elif i==7:
                    pass
                elif i==8:
                    active_neighbors_set = line[23:]
                elif i==9:
                    passive_neighbors_set = line[24:]
                i+=1

            structure.close()
        
        active_neighbors_set = json.loads(active_neighbors_set)
        passive_neighbors_set = json.loads(passive_neighbors_set)

        active_degrees = [len(elem) for elem in active_neighbors_set]
        passive_degrees = [len(elem) for elem in passive_neighbors_set]
        
        os.makedirs('AME_MC_data/Structures/Homophily_{0}/time_series_mult_{1}'.format(10*hom, mult), exist_ok=True)
        
        states = MC_simulation(mult/n_max, 0.2, active_degrees, passive_degrees, active_neighbors_set, 
                               passive_neighbors_set, sample_size=800, burn_in_time_constant = 2e6,
                              savefile = "AME_MC_data/Structures/Homophily_{0}/time_series_mult_{1}/run_{2}.npz".format(10*hom, mult, it))

## Threshold validation

In [18]:
N=200000
A=0.5
s=10

thresh_hom_mean = [[], []]
thresh_hom_stdv = [[], []]
threshs = [[0.03187005539524489, 0.03028699063867356, 0.02653055542655211, 
            0.022571923691326472, 0.019642685917530604, 0.017866416005383304, 
            0.01689796776191878, 0.016367209879100692, 0.015987936068458317, 
            0.01550920950415379, 0.014171447760062066],
           [0.014300344492904623, 0.014323495875413797, 0.014404781908378723, 
            0.014575628442389152, 0.014862990254993807, 0.01523110789126192, 
            0.01555032369334468, 0.015706013657075007, 0.01567939074563655, 
            0.015402875463826859, 0.014171447760062066]]

invphi = (np.sqrt(5) - 1) / 2

for tau in [0.3, 0.7]:
    k = 0
    tau_in = 0 if tau == 0.3 else 1
    for h in np.linspace(0, 1.0, 11):
        susc_list = []
        for it in tqdm(range(50)):
            if tau == 0.3:
                path = "AME_MC_data/Structures/Homophily_{0}/structure_{1}A.txt".format(round(10*h, 1), it)
            else:
                path = "AME_MC_data/Structures/Homophily_{0}/structure_{1}.txt".format(round(10*h, 1), it)
            with open(path, "r") as structure:
                i = 0
                for line in structure:
                    if i in [0, 1, 2, 3, 4, 5, 6, 7]:
                        pass
                    elif i==8:
                        active_neighbors_set = line[23:]
                    elif i==9:
                        passive_neighbors_set = line[24:]
                    elif i==10:
                        pass
                    elif i==11:
                        pass
                    i+=1

                structure.close()

            active_neighbors_set = json.loads(active_neighbors_set)
            passive_neighbors_set = json.loads(passive_neighbors_set)

            active_degrees = [len(elem) for elem in active_neighbors_set]
            passive_degrees = [len(elem) for elem in passive_neighbors_set]
            
            mult_0 = 0.5*threshs[tau_in][k]
            mult_1 = 1.5*threshs[tau_in][k]
            diff = mult_1 - mult_0
            while diff>1e-3:
                bound1 = mult_1 - diff*invphi
                bound2 = mult_0 + diff*invphi
                susc1 = []
                susc2 = []
        
                states1 = MC_simulation(bound1, 0.2, active_degrees, passive_degrees, active_neighbors_set, 
                                       passive_neighbors_set, sample_size=1000, burn_in_time_constant = 5e5, init_prev=0.02,
                                       min_interval=10, output_states = True)
                prevs1 = [len(elem)/N for elem in states1]
                susceptibility1 = np.var(prevs1)/np.mean(prevs1)

                states2 = MC_simulation(bound2, 0.2, active_degrees, passive_degrees, active_neighbors_set, 
                                       passive_neighbors_set, sample_size=1000, burn_in_time_constant = 5e5, init_prev=0.02,
                                       min_interval=10, output_states = True)
                prevs2 = [len(elem)/N for elem in states2]
                susceptibility2 = np.var(prevs2)/np.mean(prevs2)

                if susceptibility1 > susceptibility2:
                    mult_1 = bound2
                else:
                    mult_0 = bound1
                diff = mult_1 - mult_0
                
            susc_list.append(0.5 * (mult_0 + mult_1))

        if tau == 0.3:
            thresh_hom_mean[0].append(np.mean(susc_list))
            thresh_hom_stdv[0].append(np.sqrt(np.var(susc_list)))
        else:
            thresh_hom_mean[1].append(np.mean(susc_list))
            thresh_hom_stdv[1].append(np.sqrt(np.var(susc_list)))
        k += 1

100%|██████████| 50/50 [3:00:14<00:00, 216.30s/it]  


In [19]:
print(thresh_hom_mean)
print(thresh_hom_stdv)
#These can be used in the AME_model notebook for figure 8

[[0.031574297879849306, 0.027946319951634928, 0.022169034660336026, 0.019532570923554704, 0.01773451385646567, 0.016575385572449405, 0.015465299386159094, 0.0151791893286646, 0.014545925461497227, 0.014379386593479456, 0.013738406850098054], [0.013782234576512334, 0.013789472416140898, 0.013797375308318434, 0.01438158850558012, 0.014730464485615897, 0.01512925182509671, 0.015064436444673288, 0.015080327102976384, 0.014518118955568293, 0.014150973271955532, 0.013848971530368149]]
[[0.008004334585021037, 0.00572946890189738, 0.0038536364035616608, 0.0012895850384298058, 0.0014104096771683007, 0.0010784025187492619, 0.000591930661054599, 0.0007437453256413047, 0.00081450330277189, 0.0007344061963261547, 0.0006337978045642388], [0.0006347548240141625, 0.0005564073239797344, 0.0007255527617348988, 0.0005476698502423727, 0.0006454184423759634, 0.0010341740211245616, 0.0007245765237239385, 0.0006873066768314785, 0.0007422607522249159, 0.0007052052743110839, 0.0006336835463831968]]


## Steady-state validation

In [ ]:
N=200000
A=0.5
tau=0.6
s=10
hom = 0.0

mult_list = [i/2 for i in range(1, 17)]

group_5_prev = [[] for i in range(len(mult_list))]
group_10_prev = [[] for i in range(len(mult_list))]
group_20_prev = [[] for i in range(len(mult_list))]
group_30_prev = [[] for i in range(len(mult_list))]
group_50_prev = [[] for i in range(len(mult_list))]
group_70_prev = [[] for i in range(len(mult_list))]

for it in tqdm(range(50)):
    with open("AME_MC_data/Structures/Homophily_{0}/structure_{1}.txt".format(10*hom, it), "rt") as structure:
        i = 0
        for line in structure:
            if i==0 or i==1:
                pass
            elif i==2:
                active_node_stubs = line[20:]
            elif i==3:
                passive_node_stubs = line[21:]
            elif i==4:
                group_size = line[13:]
            elif i==5:
                pass
            elif i==6:
                active_group_stubs = line[21:]
            elif i==7:
                passive_group_stubs = line[22:]
            elif i==8:
                pass
            elif i==9:
                pass
            elif i==10:
                pass
            i+=1

        structure.close()

    active_node_stubs = json.loads(active_node_stubs)
    passive_node_stubs = json.loads(passive_node_stubs)
    group_size = json.loads(group_size)
    active_group_stubs = json.loads(active_group_stubs)
    passive_group_stubs = json.loads(passive_group_stubs)
    active_group_indices, passive_group_indices =  get_group_indices(active_group_stubs, passive_group_stubs)

    number_of_groups = len(group_size)
    groups_5 = [i for i in range(number_of_groups) if group_size[i] == 5]
    groups_10 = [i for i in range(number_of_groups) if group_size[i] == 10]
    groups_20 = [i for i in range(number_of_groups) if group_size[i] == 20]
    groups_30 = [i for i in range(number_of_groups) if group_size[i] == 30]
    groups_50 = [i for i in range(number_of_groups) if group_size[i] == 50]
    groups_70 = [i for i in range(number_of_groups) if group_size[i] == 70]

    group_5_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_5]
    group_10_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_10]
    group_20_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_20]
    group_30_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_30]
    group_50_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_50]
    group_70_nodes = [group_nodes(group, active_node_stubs, active_group_indices, passive_node_stubs, passive_group_indices) for group in groups_70]
    
    mult_it = 0
    for mult in [i/2 for i in range(1, 17)]:
        data = np.load("AME_MC_data/Structures/Homophily_{0}/time_series_mult_{1}/run_{2}.npz"
                            .format(10*hom, mult, it))
        num_samples = len(data.files)
        for i in range(num_samples):
            infected_nodes = set(data[f"s{i}"])
            inf_group_5_nodes = [infected_nodes.intersection(set(elem)) for elem in group_5_nodes]
            inf_group_10_nodes = [infected_nodes.intersection(set(elem)) for elem in group_10_nodes]
            inf_group_20_nodes = [infected_nodes.intersection(set(elem)) for elem in group_20_nodes]
            inf_group_30_nodes = [infected_nodes.intersection(set(elem)) for elem in group_30_nodes]
            inf_group_50_nodes = [infected_nodes.intersection(set(elem)) for elem in group_50_nodes]
            inf_group_70_nodes = [infected_nodes.intersection(set(elem)) for elem in group_70_nodes]

            prev_5 = [len(inf_group_5_nodes[i])/len(group_5_nodes[i]) for i in range(len(group_5_nodes))]
            prev_10 = [len(inf_group_10_nodes[i])/len(group_10_nodes[i]) for i in range(len(group_10_nodes))]
            prev_20 = [len(inf_group_20_nodes[i])/len(group_20_nodes[i]) for i in range(len(group_20_nodes))]
            prev_30 = [len(inf_group_30_nodes[i])/len(group_30_nodes[i]) for i in range(len(group_30_nodes))]
            prev_50 = [len(inf_group_50_nodes[i])/len(group_50_nodes[i]) for i in range(len(group_50_nodes))]
            prev_70 = [len(inf_group_70_nodes[i])/len(group_70_nodes[i]) for i in range(len(group_70_nodes))]

            del inf_group_5_nodes
            del inf_group_10_nodes
            del inf_group_20_nodes
            del inf_group_30_nodes
            del inf_group_50_nodes
            del inf_group_70_nodes

            if len(prev_5) > 0:
                group_5_prev[mult_it].append(np.mean(prev_5))
            if len(prev_10) > 0:
                group_10_prev[mult_it].append(np.mean(prev_10))
            if len(prev_20) > 0:
                group_20_prev[mult_it].append(np.mean(prev_20))
            if len(prev_30) > 0:
                group_30_prev[mult_it].append(np.mean(prev_30))
            if len(prev_50) > 0:
                group_50_prev[mult_it].append(np.mean(prev_50))
            if len(prev_70) > 0:
                group_70_prev[mult_it].append(np.mean(prev_70))
                
        mult_it += 1

In [ ]:
A = 0.5
tau = 0.6
hom_list = [0.4]

s_act_hom_list, s_inact_hom_list, c_act_hom_list, c_inact_hom_list, group_prev_hom_list, global_prev_hom_list = [], [], [], [], [], []

show_parameters = True
for hom in hom_list:
    with open ("AME_MC_data/Structures/Homophily_4.0/Theoretical_QS_state_corr.txt", "rt") as data:
        i = 0
        for line in data:
            if i==0 or i==1:
                if show_parameters:
                    print(line) #Shows parameters of the data file for cross-checking if show_parameters = True
            elif i==2:
                s_m_act_text = line[8:]
            elif i==3:
                s_m_inact_text = line[8:]
            elif i==4:
                c_act_text = line[9:]
            elif i==5:
                c_inact_text = line[9:]
            elif i==6:
                group_prev_text = line[19:]
            elif i==7:
                global_prev_text = line[20:]
            i+=1
        s_act_hom = ast.literal_eval(s_m_act_text)
        s_inact_hom = ast.literal_eval(s_m_inact_text)
        c_act_hom = ast.literal_eval(c_act_text)
        c_inact_hom = ast.literal_eval(c_inact_text)
        group_prev_hom = ast.literal_eval(group_prev_text)
        global_prev_hom = ast.literal_eval(global_prev_text)
        s_act_hom_list.append(s_act_hom)
        s_inact_hom_list.append(s_inact_hom)
        c_act_hom_list.append(c_act_hom)
        c_inact_hom_list.append(c_inact_hom)
        group_prev_hom_list.append(group_prev_hom)
        global_prev_hom_list.append(global_prev_hom)
        
        data.close()

In [ ]:
def active_group_prevalence(n, c_1):
    if n<n_min or n>n_max:
        return 0
    else:
        c_n = c_1[n-n_min]
        terms = [(i/n)*c_n[i] for i in range(0, n+1)]
        return np.sum(terms)
    
def gamma(n, steepness, hom, inf, D):  #Probability that a randomly chosen group of size n is active
    array_sum = [P_n_n_act(n_act, n, steepness, inf)*P_n_act(n_act, n, hom, D) for n_act in range(n+1)]
    return np.sum(array_sum)

#This cell must be run if the following three cells are to be run
n_list = [5, 10, 20, 30, 50, 70]
act_group_prev_hom = []
act_group_prev = []

for n in [5, 10, 20, 30, 50, 70]:
    act_group_prev_n = []
    for c in c_act_hom_list[0]:
        act_group_prev_n.append(np.array(active_group_prevalence(n, c))) 
    act_group_prev.append(np.array(act_group_prev_n))
act_group_prev_hom.append(np.array(act_group_prev))
act_group_prev_hom = np.array(act_group_prev_hom)

inact_group_prev_hom = []
inact_group_prev = []

for n in [5, 10, 20, 30, 50, 70]:
    inact_group_prev_n = []
    for c in c_inact_hom_list[0]:
        inact_group_prev_n.append(np.array(active_group_prevalence(n, c)))
    inact_group_prev.append(np.array(inact_group_prev_n))
inact_group_prev_hom.append(np.array(inact_group_prev))
inact_group_prev_hom = np.array(inact_group_prev_hom)

group_prev_n_hom = np.array([gamma(n_list[i], 10, hom, tau, A)*act_group_prev_hom[0][i] + (1-gamma(n_list[i], 10, hom, tau, A))*inact_group_prev_hom[0][i] for i in range(6)])

In [ ]:
n_list = [5, 10, 20, 30, 50, 70]
act_group_prev_hom = []
act_group_prev = []

for n in [5, 10, 20, 30, 50, 70]:
    act_group_prev_n = []
    for c in c_act_hom_list[0]:
        act_group_prev_n.append(np.array(active_group_prevalence(n, c))) 
    act_group_prev.append(np.array(act_group_prev_n))
act_group_prev_hom.append(np.array(act_group_prev))
act_group_prev_hom = np.array(act_group_prev_hom)

inact_group_prev_hom = []
inact_group_prev = []

for n in [5, 10, 20, 30, 50, 70]:
    inact_group_prev_n = []
    for c in c_inact_hom_list[0]:
        inact_group_prev_n.append(np.array(active_group_prevalence(n, c)))
    inact_group_prev.append(np.array(inact_group_prev_n))
inact_group_prev_hom.append(np.array(inact_group_prev))
inact_group_prev_hom = np.array(inact_group_prev_hom)

group_prev_n_hom_2 = np.array([gamma(n_list[i], 10, hom, tau, A)*act_group_prev_hom[0][i] + (1-gamma(n_list[i], 10, hom, tau, A))*inact_group_prev_hom[0][i] for i in range(6)])

In [ ]:
mult_list = [i/2 for i in range(1, 17)]
mean_prev_5 = [np.mean(group_5_prev[mult_it]) for mult_it in range(len(mult_list))]
mean_prev_10 = [np.mean(group_10_prev[mult_it]) for mult_it in range(len(mult_list))]
mean_prev_20 = [np.mean(group_20_prev[mult_it]) for mult_it in range(len(mult_list))]
mean_prev_30 = [np.mean(group_30_prev[mult_it]) for mult_it in range(len(mult_list))]
mean_prev_50 = [np.mean(group_50_prev[mult_it]) for mult_it in range(len(mult_list))]
mean_prev_70 = [np.mean(group_70_prev[mult_it]) for mult_it in range(len(mult_list))]

stdv_prev_5 = [np.sqrt(np.var(group_5_prev[mult_it])) for mult_it in range(len(mult_list))]
stdv_prev_10 = [np.sqrt(np.var(group_10_prev[mult_it])) for mult_it in range(len(mult_list))]
stdv_prev_20 = [np.sqrt(np.var(group_20_prev[mult_it])) for mult_it in range(len(mult_list))]
stdv_prev_30 = [np.sqrt(np.var(group_30_prev[mult_it])) for mult_it in range(len(mult_list))]
stdv_prev_50 = [np.sqrt(np.var(group_50_prev[mult_it])) for mult_it in range(len(mult_list))]
stdv_prev_70 = [np.sqrt(np.var(group_70_prev[mult_it])) for mult_it in range(len(mult_list))]

In [ ]:
plt.rcParams["figure.figsize"] = (10,8)

mult_list_th = [i/(4*4) for i in range(1, 4*32+1)]
mult_list = [i/(2) for i in range(1, 17)]
means_array = [mean_prev_5, mean_prev_10, mean_prev_20, mean_prev_30, mean_prev_50, mean_prev_70]
stdv_array = [stdv_prev_5, stdv_prev_10, stdv_prev_20, stdv_prev_30, stdv_prev_50, stdv_prev_70]

i=0
for n in [5, 10, 20, 30, 50, 70]:
    group_prev_1 = group_prev_n_hom[i]
    group_prev_2 = group_prev_n_hom_2[i]
    #plt.plot(mult_list_th, group_prev_1, color='black')
    plt.plot(mult_list_th, group_prev_2, color='red')
    plt.errorbar(mult_list, means_array[i], yerr=stdv_array[i], color=colors_7[i], marker='o', linestyle='None')
    #plt.plot(mult_list, means_array[i], color=colors_7[i], marker='o', linestyle='None')
    i += 1

#plt.savefig('Figures/MC_validation_bifurcation_diagram', dpi=400)
#fig_data = open('Figures/MC_validation_bifurcation_data.npy', 'wb')
#np.save(fig_data, mult_list_th)
#np.save(fig_data, mult_list)
#np.save(fig_data, means_array)
#np.save(fig_data, stdv_array)
#np.save(fig_data, group_prev_n_hom)
#plt.axvline(1.283, color='black')
#plt.xlim([1.1, 1.4])
#plt.ylim([0, 0.003])
plt.show()

In [ ]:
means_array = [mean_prev_5, mean_prev_10, mean_prev_20, mean_prev_30, mean_prev_50, mean_prev_70]
stdv_array = [stdv_prev_5, stdv_prev_10, stdv_prev_20, stdv_prev_30, stdv_prev_50, stdv_prev_70]
print(means_array)
print(stdv_array)